# Teaching Monster — Colab Pro+ GPU Notebook

**Runtime**: GPU (A100 on Pro+), High-RAM  

## Before first run — set Secrets (one-time setup)

Click the **key icon (🔑)** in the left sidebar → Add two secrets:

| Name | Value |
|---|---|
| `GEMINI_API_KEY` | your key from aistudio.google.com |
| `NGROK_TOKEN` | your token from dashboard.ngrok.com |

Enable **"Notebook access"** for both. After that, `Run all` works fully unattended.

> Also set **Runtime → Change runtime type → GPU → A100**, enable High-RAM.

## 0 · GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — change runtime to GPU first!"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__}")

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone repository from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/ragiokay/TeachingMonster-released-main.git"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

## 3 · Install system packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libreoffice poppler-utils
print("System packages ready.")

## 4 · Install Python dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements-colab-gpu.txt
print("Python packages ready.")

## 5 · Configure Gemini API key

Reads from Colab Secrets automatically. Falls back to manual input if not set.

In [ ]:
import getpass
from pathlib import Path

try:
    from google.colab import userdata
    key = userdata.get('GEMINI_API_KEY')
    if not key:
        raise KeyError
    print("GEMINI_API_KEY loaded from Colab Secrets.")
except Exception:
    key = getpass.getpass("GEMINI_API_KEY not in Secrets — paste it here: ")

Path("config/.env").write_text(f"GEMINI_API_KEY={key}\n", encoding="utf-8")
print("config/.env written.")

## 6 · Verify Gemini connectivity

In [ ]:
import yaml
from src.config_schema import AppConfig
from src.gemini_client import GeminiClient

cfg = AppConfig(**yaml.safe_load(open("config/default.yaml", encoding="utf-8")))
client = GeminiClient(cfg.llm)
client.load()
print("Gemini says:", client.generate("Reply with OK only.")[:80])

---
# API Server Mode (Competition)

## How it works

The judge's platform sends:
```json
POST /v1/video/generate
{ "request_id": "job-001", "course_requirement": "...", "student_persona": "..." }
```

Your server generates the video, uploads it to **Google Drive** (48+ hour links), and responds:
```json
{ "video_url": "https://drive.google.com/uc?export=download&confirm=t&id=...", "subtitle_url": "...", "supplementary_url": null }
```

**Run cells 7 → 12 in order.**

## 7 · Authenticate Google Drive API

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Google Drive API authenticated.")

## 8 · Create public output folder in Drive

In [ ]:
import os
from googleapiclient.discovery import build
from google.auth import default

creds, _ = default()
drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)

folder = drive_service.files().create(
    body={"name": "TeachingMonster-Competition-Videos",
          "mimeType": "application/vnd.google-apps.folder"},
    fields="id",
).execute()
DRIVE_FOLDER_ID = folder["id"]

drive_service.permissions().create(
    fileId=DRIVE_FOLDER_ID,
    body={"type": "anyone", "role": "reader"},
).execute()

os.environ["DRIVE_OUTPUT_FOLDER_ID"] = DRIVE_FOLDER_ID
print(f"Drive folder: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"DRIVE_OUTPUT_FOLDER_ID={DRIVE_FOLDER_ID}")

## 9 · Configure ngrok

Reads token from Colab Secrets automatically. Falls back to manual input if not set.

In [ ]:
!pip -q install pyngrok

import getpass
from pyngrok import conf

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_TOKEN')
    if not ngrok_token:
        raise KeyError
    print("NGROK_TOKEN loaded from Colab Secrets.")
except Exception:
    ngrok_token = getpass.getpass("NGROK_TOKEN not in Secrets — paste it here: ")

conf.get_default().auth_token = ngrok_token
print("ngrok ready.")

## 10 · Start API server + ngrok tunnel

**Models download on first run (~35 GB, 15–25 min). Watch Cell 11 for progress.**

In [ ]:
import subprocess, os
from pyngrok import ngrok

# Kill any leftover server or tunnel from a previous run of this cell
!fuser -k 8000/tcp 2>/dev/null || true
ngrok.kill()

os.environ.pop("TTS_FAST_MODE", None)
os.environ.pop("CURSOR_FAST_MODE", None)

tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url
os.environ["BASE_URL"] = PUBLIC_URL

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.server:app",
     "--host", "0.0.0.0", "--port", "8000"],
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("=" * 60)
print(f"ENDPOINT:  POST {PUBLIC_URL}/v1/video/generate")
print(f"HEALTH:    GET  {PUBLIC_URL}/health")
print("=" * 60)
print("Run Cell 11 to watch logs. Wait for 'Pipeline loaded successfully!'")

## 11 · Watch server logs — wait for `Pipeline loaded successfully!`

In [ ]:
import io
for line in io.TextIOWrapper(server_proc.stdout, encoding="utf-8"):
    print(line, end="")
    if "Pipeline loaded successfully" in line:
        print("\n>>> Server ready — give judges the endpoint URL. <<<")
        break

## 12 · Dry-run test

In [ ]:
import requests as req

resp = req.post(
    f"{PUBLIC_URL}/v1/video/generate",
    json={"request_id": "dry-run-001",
          "course_requirement": "Test",
          "student_persona": "Test"},
    headers={"X-Dry-Run": "true"},
    timeout=10,
)
assert resp.status_code == 200, f"FAILED {resp.status_code}: {resp.text}"
print("Dry-run OK:", resp.json())
print()
print("Give judges:")
print(f"  POST {PUBLIC_URL}/v1/video/generate")

---
### Troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Enable High-RAM runtime, rerun from Cell 0 |
| Port 8000 already in use | Re-run Cell 10 (it kills the old process first) |
| ngrok 401 | Check NGROK_TOKEN in Secrets |
| Drive upload fails | Re-run Cell 7 (re-authenticate) |
| Session expires mid-competition | Rerun Cells 0–12; previous Drive video_urls remain valid |

### Concurrent requests
The server serialises all requests with an `asyncio.Lock`. Concurrent judge requests queue automatically — no race conditions.